# DLinear 재현 (Colab)
논문: *Are Transformers Effective for Time Series Forecasting?* (AAAI 2023) · 코드: cure-lab/LTSF-Linear

**먼저 GPU 런타임에 연결:**
- 브라우저 Colab: 런타임 → 런타임 유형 변경 → GPU (T4면 충분)
- VS Code: 오른쪽 위 `Select Kernel` → `Colab` → `New Colab Server` → GPU(T4 이상) 선택 → 처음 한 번 구글 로그인

DLinear는 아주 작아서 CPU로도 돌지만 Traffic/Electricity는 GPU가 훨씬 빠르다.
VS Code에서도 코드는 내 PC가 아니라 **Colab 서버의 `/content`** 에서 돈다 (데이터·결과도 거기 생김). 서버 파일은 왼쪽 Colab 아이콘의 content 뷰에서 볼 수 있다.

In [ ]:
# 0. GPU 확인
!nvidia-smi
import torch, numpy, pandas
print(torch.__version__, numpy.__version__, pandas.__version__, 'cuda:', torch.cuda.is_available())

In [ ]:
# 1. 공식 코드 받기
%cd /content
!git clone https://github.com/cure-lab/LTSF-Linear.git
%cd /content/LTSF-Linear

## 2. 최신 Colab 호환 패치
- 레포는 `torch==1.9`, Python 3.6 기준. **requirements.txt는 설치하지 않는다** (Colab 기본 torch 그대로 사용).
- NumPy 2.0에서 `np.Inf`가 삭제돼서 `EarlyStopping`에서 바로 죽는다 → `np.inf`로 바꾼다. (실제로 최신 버전에서 돌려서 확인한 에러)
- DataLoader 기본 `num_workers=10` → Colab CPU 개수(2)에 맞춘다. 결과에는 영향 없음, 경고만 줄어든다.

In [ ]:
!sed -i 's/np\.Inf/np.inf/' utils/tools.py
!sed -i "s/default=10, help='data loader num workers'/default=2, help='data loader num workers'/" run_longExp.py
!grep -n "np.inf" utils/tools.py
!grep -n "num_workers" run_longExp.py

## 3. 데이터 준비
- ETT 4종은 원 저장소(GitHub)에서 바로 받는다.
- 나머지(Electricity, Exchange, Traffic, Weather, ILI)는 Autoformer 저자 구글 드라이브 폴더 → `gdown`.
- gdown이 '다운로드 한도' 에러를 내면: 링크를 브라우저로 열어 zip을 받아 내 드라이브에 올리고, `drive.mount` 후 csv를 `dataset/`으로 복사.

In [ ]:
!mkdir -p dataset
for f in ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2']:
    !wget -q -O dataset/{f}.csv https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/{f}.csv
!ls -lh dataset

In [ ]:
!pip -q install gdown
!gdown --folder https://drive.google.com/drive/folders/1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy -O raw_data
!find raw_data -name "*.csv" -exec cp -n {} dataset/ \;
!ls -lh dataset   # 필요한 파일: electricity, exchange_rate, national_illness, traffic, weather .csv

## 4. 첫 실험: ETTh1, L=336 → T=96
스크립트(`scripts/EXP-LongForecasting/Linear/etth1.sh`)의 첫 번째 줄과 같은 설정. 로그를 화면에 그대로 띄운다.
마지막 줄 `mse:…, mae:…`가 논문 Table 2의 **0.375 / 0.399** 근처면 성공 (±0.01 정도는 시드·하드웨어 차이).

In [ ]:
!python -u run_longExp.py --is_training 1 --root_path ./dataset/ --data_path ETTh1.csv \
  --model_id ETTh1_336_96 --model DLinear --data ETTh1 --features M \
  --seq_len 336 --pred_len 96 --enc_in 7 --des Exp --itr 1 --batch_size 32 --learning_rate 0.005

## 5. 논문 Table 2 (DLinear, 다변량) 전체 돌리기
스크립트 하나 = 데이터셋 하나 × horizon 4개. 로그는 `logs/LongForecasting/`에, 지표는 `result.txt`에 쌓인다.
ETT·Exchange·ILI·Weather는 금방, Electricity·Traffic은 오래 걸린다 (특히 Traffic). 필요한 것만 남기고 돌려도 된다.

In [ ]:
SCRIPTS = ['etth1', 'etth2', 'ettm1', 'ettm2', 'exchange_rate', 'ili', 'weather', 'electricity', 'traffic']
import subprocess, time
for s in SCRIPTS:
    t = time.time()
    subprocess.run(['sh', f'scripts/EXP-LongForecasting/Linear/{s}.sh'], check=True)
    print(f'{s}: {time.time() - t:.0f}s')

## 6. 결과 모아서 논문 값과 비교
`result.txt`는 [실험 이름 한 줄, `mse:…, mae:…` 한 줄, 빈 줄] 반복. 이름에서 데이터셋·horizon·반복번호를 뽑는다.

In [ ]:
import re, pandas as pd

PAPER = {  # 논문 Table 2, DLinear (MSE, MAE)
 'ETTh1': {96:(0.375,0.399),192:(0.405,0.416),336:(0.439,0.443),720:(0.472,0.490)},
 'ETTh2': {96:(0.289,0.353),192:(0.383,0.418),336:(0.448,0.465),720:(0.605,0.551)},
 'ETTm1': {96:(0.299,0.343),192:(0.335,0.365),336:(0.369,0.386),720:(0.425,0.421)},
 'ETTm2': {96:(0.167,0.260),192:(0.224,0.303),336:(0.281,0.342),720:(0.397,0.421)},
 'Exchange': {96:(0.081,0.203),192:(0.157,0.293),336:(0.305,0.414),720:(0.643,0.601)},
 'weather': {96:(0.176,0.237),192:(0.220,0.282),336:(0.265,0.319),720:(0.323,0.362)},
 'Electricity': {96:(0.140,0.237),192:(0.153,0.249),336:(0.169,0.267),720:(0.203,0.301)},
 'traffic': {96:(0.410,0.282),192:(0.423,0.287),336:(0.436,0.296),720:(0.466,0.315)},
 'national_illness': {24:(2.215,1.081),36:(1.963,0.963),48:(2.130,1.024),60:(2.368,1.096)},
}

def load_results(path='result.txt', model='DLinear'):
    rows = []
    for block in open(path).read().strip().split('\n\n'):
        lines = [l.strip() for l in block.strip().split('\n') if l.strip()]
        if len(lines) < 2 or f'_{model}_' not in lines[0]:
            continue
        name = lines[0]
        model_id = name.split(f'_{model}_')[0]
        ds = re.sub(r'_\d+_\d+$', '', model_id)
        pl = int(re.search(r'_pl(\d+)_', name).group(1))
        run = int(re.search(r'_(\d+)$', name).group(1))
        mse = float(re.search(r'mse:([\d.eE+-]+)', lines[1]).group(1))
        mae = float(re.search(r'mae:([\d.eE+-]+)', lines[1]).group(1))
        p = PAPER.get(ds, {}).get(pl, (None, None))
        rows.append(dict(dataset=ds, pred_len=pl, run=run, mse=mse, mae=mae, paper_mse=p[0], paper_mae=p[1]))
    df = pd.DataFrame(rows)
    df['mse_diff'] = df['mse'] - df['paper_mse']
    return df

res = load_results()
res.groupby(['dataset', 'pred_len'])[['mse', 'mae', 'paper_mse', 'paper_mae', 'mse_diff']].mean().round(3)

## 7. 시드 편차 확인 (중요!)
`run_longExp.py`는 시작할 때 seed=2021을 한 번만 고정하고, `--itr N`이면 같은 실험을 N번 이어서 돌린다(2번째부터는 난수 상태가 달라서 결과도 다름).
ETTh1처럼 작은 데이터는 학습이 2~3 epoch만에 early stop돼서 **시드에 따라 MSE가 꽤 흔들린다.** 한 번 결과로 판단하지 말고 3번 평균을 보자.

In [ ]:
!python -u run_longExp.py --is_training 1 --root_path ./dataset/ --data_path ETTh1.csv \
  --model_id ETTh1_336_336 --model DLinear --data ETTh1 --features M \
  --seq_len 336 --pred_len 336 --enc_in 7 --des Exp --itr 3 --batch_size 32 --learning_rate 0.005 | grep "mse:"
res = load_results()
res[(res.dataset == 'ETTh1') & (res.pred_len == 336)][['run', 'mse', 'mae', 'paper_mse']]

## 8. 예측 그래프 직접 그려보기
체크포인트를 불러와 test 셋 한 샘플을 그린다 (값은 정규화된 스케일).

In [ ]:
import glob, argparse, torch, matplotlib.pyplot as plt
from data_provider.data_factory import data_provider
from models import DLinear

args = argparse.Namespace(data='ETTh1', root_path='./dataset/', data_path='ETTh1.csv', features='M', target='OT',
                          freq='h', embed='timeF', train_only=False, seq_len=336, label_len=48, pred_len=96,
                          batch_size=32, num_workers=2, enc_in=7, individual=False)
test_set, _ = data_provider(args, 'test')
model = DLinear.Model(args)
ckpt = sorted(glob.glob('checkpoints/ETTh1_336_96_DLinear*/checkpoint.pth'))[0]
model.load_state_dict(torch.load(ckpt, map_location='cpu'))
model.eval()

i, ch = 500, -1              # test 샘플 번호, 채널(-1 = OT)
x, y, _, _ = test_set[i]
with torch.no_grad():
    pred = model(torch.tensor(x).float().unsqueeze(0))[0].numpy()
L, T = args.seq_len, args.pred_len
plt.figure(figsize=(12, 4))
plt.plot(range(L), x[:, ch], label='input (336)')
plt.plot(range(L, L + T), y[-T:, ch], label='truth (96)')
plt.plot(range(L, L + T), pred[:, ch], label='DLinear')
plt.legend(); plt.title('ETTh1 OT, test sample'); plt.show()

## 9. 가중치 시각화 (논문 Figure 7 따라하기)
논문은 가중치를 $1/L$로 초기화해야 패턴이 깔끔하게 보인다고 한다. `models/DLinear.py`에 주석 처리된 두 줄(individual=False 쪽)을 살려서 다시 학습한 뒤 그린다.

In [ ]:
!sed -i 's/^            # self.Linear_\(Seasonal\|Trend\).weight/            self.Linear_\1.weight/' models/DLinear.py
!grep -n "1/self.seq_len" models/DLinear.py
!python -u run_longExp.py --is_training 1 --root_path ./dataset/ --data_path ETTh1.csv \
  --model_id ETTh1_336_96_init --model DLinear --data ETTh1 --features M \
  --seq_len 336 --pred_len 96 --enc_in 7 --des Exp --itr 1 --batch_size 32 --learning_rate 0.005 | grep "mse:"

sd = torch.load(sorted(glob.glob('checkpoints/ETTh1_336_96_init_DLinear*/checkpoint.pth'))[0], map_location='cpu')
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, k in zip(axes, ['Linear_Seasonal.weight', 'Linear_Trend.weight']):
    W = sd[k].numpy(); v = abs(W).max()
    im = ax.imshow(W, cmap='RdBu_r', vmin=-v, vmax=v, aspect='auto')
    ax.set_title(k); ax.set_xlabel('input step (335 = most recent)'); ax.set_ylabel('output step')
    fig.colorbar(im, ax=ax)
plt.show()

## 10. (선택) 결과를 구글 드라이브에 보관
Colab 런타임이 끊기면 `/content`가 날아간다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/DLinear_repro
!cp -r result.txt logs /content/drive/MyDrive/DLinear_repro/